# TUẦN 3 & TUẦN 4: MÔ HÌNH HỌC SÂU BI-LSTM & TINH CHỈNH PHOBERT
## (DEEP LEARNING ARCHITECTURES: BI-LSTM & PHOBERT FINE-TUNING)

**Dự án:** Hệ thống Phân tích Cảm xúc Đánh giá Khách hàng Tiếng Việt (Vietnamese Review Sentiment Analysis)  
**Học phần:** Artificial Intelligence Capstone 2025 • Viện Nghiên cứu & Đào tạo Việt - Anh (VNUK)  
**Giảng viên hướng dẫn:** TS. Trần Thế Vũ (`vu.tran@vnuk.edu.vn`)  
**Kỹ sư phụ trách:** **Châu Tuấn Kiệt — Senior ML Engineer**  
**Nhóm sinh viên thực hiện:** Thái Bảo (Leader), Tuấn Kiệt, Đức Minh, Đình Hiếu, Huy Hoàng  

---

### 🎯 Mục tiêu Thử nghiệm & Câu hỏi Nghiên cứu (Research Questions):
1. **RQ2 (Kiến trúc Mô hình):** Cơ chế đọc ngữ cảnh 2 chiều (Forward + Backward) của mạng **Bi-LSTM** nắm bắt ngữ nghĩa câu đảo và câu phủ định đuôi vượt trội hơn mô hình Baseline (TF-IDF + Logistic Regression) như thế nào?
2. **RQ3 (Xử lý Mất cân bằng Nhãn):** Ứng dụng hàm mất mát có trọng số **Weighted Cross-Entropy Loss** ($w_c = \frac{N}{K \cdot N_c}$) cải thiện chỉ số **Macro F1-score** trên các phân lớp thiểu số (Negative ~21%, Neutral ~15%) ra sao so với Standard Cross-Entropy?
3. **SOTA Benchmark:** Thiết lập khung tinh chỉnh mô hình Transformer tiền huấn luyện **VinAI PhoBERT-base-v2** làm mốc trần hiệu năng tối đa.

## 1. THIẾT LẬP MÔI TRƯỜNG & HẠT GIỐNG TÁI LẬP (REPRODUCIBILITY)

In [ ]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# Cấu hình đường dẫn root để import src modules
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Hạt giống ngẫu nhiên toàn cục theo đặc tả kỹ thuật
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Đã thiết lập hạt giống ngẫu nhiên: {RANDOM_SEED}")
print(f"🖥️ Thiết bị tính toán được phân bổ: {device}")

## 2. NẠP DỮ LIỆU ĐÃ TIỀN XỬ LÝ & PHÂN TẦNG (STRATIFIED SPLITS)

In [ ]:
from src.preprocess import split_and_save_stratified

path_cleaned_csv = ROOT_DIR / "data" / "processed" / "shopee_reviews_cleaned.csv"
output_dir = ROOT_DIR / "data" / "processed"

# Đảm bảo đã có 3 tập train.csv (70%), val.csv (15%), test.csv (15%)
train_csv = output_dir / "train.csv"
val_csv = output_dir / "val.csv"
test_csv = output_dir / "test.csv"

if not (train_csv.exists() and val_csv.exists() and test_csv.exists()):
    print("⏳ Đang thực hiện phân chia dữ liệu phân tầng Stratified 70/15/15...")
    df_train, df_val, df_test = split_and_save_stratified(
        input_csv_path=str(path_cleaned_csv),
        output_dir=str(output_dir),
        train_size=0.70,
        val_size=0.15,
        test_size=0.15,
        random_state=RANDOM_SEED
    )
else:
    print("📂 Nạp trực tiếp 3 tập phân tầng có sẵn:")
    df_train = pd.read_csv(train_csv)
    df_val = pd.read_csv(val_csv)
    df_test = pd.read_csv(test_csv)

print(f"- Tập Huấn luyện (Train set 70%):     {len(df_train):,} mẫu")
print(f"- Tập Thẩm định (Validation set 15%): {len(df_val):,} mẫu")
print(f"- Tập Kiểm thử (Test set 15%):        {len(df_test):,} mẫu")
print(f"- Tổng cộng:                          {len(df_train) + len(df_val) + len(df_test):,} mẫu")

## 3. XÂY DỰNG BỘ TỪ ĐIỂN (VOCABULARY) & PYTORCH DATASET

Theo chuẩn phòng chống **Data Leakage (Rò rỉ Dữ liệu)**, từ điển từ vựng chỉ được phép học trên **tập Train** (`df_train['processed_content']`).

In [ ]:
from src.dataset import (
    Vocabulary,
    SentimentDataset,
    create_data_loader,
    build_vocab_and_datasets,
    MAX_SEQUENCE_LENGTH,
)

# Xây dựng từ điển với min_freq = 2 (loại bỏ từ hiếm)
vocab, train_dataset, val_dataset, test_dataset = build_vocab_and_datasets(
    train_df=df_train,
    val_df=df_val,
    test_df=df_test,
    text_column="processed_content",
    label_column="label_id",
    min_freq=2,
    max_length=MAX_SEQUENCE_LENGTH,
)

print(f"📚 Kích thước bộ từ điển Vocabulary: {len(vocab):,} từ")
print(f"   - Token đệm:  '{vocab.lookup_token(0)}' (ID: 0)")
print(f"   - Token lạ:   '{vocab.lookup_token(1)}' (ID: 1)")
print(f"   - 5 từ phổ biến: {[vocab.lookup_token(i) for i in range(2, 7)]}")

# Khởi tạo DataLoader cho cả 3 tập
BATCH_SIZE = 64
train_loader = create_data_loader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = create_data_loader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = create_data_loader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Kiểm tra mẫu lô đầu tiên
first_batch = next(iter(train_loader))
print(f"\n📦 Kích thước lô (Batch shape): {first_batch['input_ids'].shape}")
print(f"   Độ dài thực tế (Lengths):   {first_batch['lengths'][:5].tolist()}")
print(f"   Nhãn tương ứng (Labels):    {first_batch['labels'][:5].tolist()}")

## 4. THIẾT LẬP KIẾN TRÚC MẠNG HỌC SÂU BI-LSTM

Kiến trúc được triển khai chi tiết trong `src/models.py` gồm:
- **Embedding Layer:** $V \times 128$ với `padding_idx = 0`
- **2-Layer Bidirectional LSTM:** `hidden_dim = 128` mỗi hướng, `dropout = 0.3`
- **Concatenation:** Ghép nối hidden state cuối cùng của chiều xuôi và chiều ngược thành vector $256$ chiều
- **Output Linear:** $256 \rightarrow 3$ nhãn cảm xúc

In [ ]:
from src.models import BiLSTMClassifier

EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 2
OUTPUT_DIM = 3
DROPOUT_RATE = 0.3

model_bilstm = BiLSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    num_layers=NUM_LAYERS,
    dropout_rate=DROPOUT_RATE,
).to(device)

print(model_bilstm)
total_params = sum(p.numel() for p in model_bilstm.parameters() if p.requires_grad)
print(f"\n🔥 Tổng số tham số có thể huấn luyện (Trainable parameters): {total_params:,}")

## 5. THIẾT KẾ WEIGHTED CROSS-ENTROPY LOSS (XỬ LÝ LỆCH NHÃN RQ3)

Do nhãn POS chiếm tỷ trọng áp đảo (~64%), mô hình thông thường dễ bị bẫy thiên vị (Majority Bias).
Ta áp dụng công thức phạt trọng số nghịch đảo tần suất:

$$w_c = \frac{N}{K \cdot N_c}$$

Trong đó $N = 18.520$, $K = 3$, $N_c$ là số mẫu lớp $c$ trên tập Train.

In [ ]:
from src.models import compute_class_weights, get_weighted_criterion

train_labels = df_train['label_id'].values
class_weights = compute_class_weights(train_labels, num_classes=3, device=device)

print("⚖️ Trọng số phạt phân bổ lớp (Class Weights):")
print(f"   - Lớp 0 (NEG - Tiêu cực):  {class_weights[0]:.4f}")
print(f"   - Lớp 1 (NEU - Trung tính): {class_weights[1]:.4f}")
print(f"   - Lớp 2 (POS - Tích cực):  {class_weights[2]:.4f}")

# Khởi tạo hàm mất mát có trọng số và hàm tối ưu hóa Adam
weighted_criterion = get_weighted_criterion(class_weights=class_weights, device=device)
standard_criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_bilstm.parameters(), lr=1e-3, weight_decay=1e-5)

## 6. HUẤN LUYỆN MÔ HÌNH VỚI EARLY STOPPING & GIÁM SÁT LOSS

In [ ]:
from src.models import train_one_epoch, evaluate_model, save_checkpoint

EPOCHS = 5
best_val_acc = 0.0
history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": []
}

print(f"🚀 Bắt đầu huấn luyện mô hình Bi-LSTM ({EPOCHS} Epochs)...")
for epoch in range(1, EPOCHS + 1):
    t_loss, t_acc = train_one_epoch(
        model=model_bilstm,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=weighted_criterion,
        device=device,
        clip_grad=1.0,
    )
    
    v_loss, v_acc, _, _ = evaluate_model(
        model=model_bilstm,
        dataloader=val_loader,
        criterion=weighted_criterion,
        device=device,
    )
    
    history["train_loss"].append(t_loss)
    history["train_acc"].append(t_acc)
    history["val_loss"].append(v_loss)
    history["val_acc"].append(v_acc)
    
    print(f"Epoch [{epoch}/{EPOCHS}] | Train Loss: {t_loss:.4f} - Train Acc: {t_acc*100:.2f}% | Val Loss: {v_loss:.4f} - Val Acc: {v_acc*100:.2f}%")
    
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        ckpt_path = ROOT_DIR / "models" / "best_bilstm_model.pt"
        save_checkpoint(
            model=model_bilstm,
            optimizer=optimizer,
            epoch=epoch,
            metrics={"val_acc": v_acc, "val_loss": v_loss},
            save_path=ckpt_path
        )
        print(f"  🌟 Đã lưu checkpoint xuất sắc nhất tại: {ckpt_path}")

## 7. ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP & MA TRẬN NHẦM LẪN (RQ2 BENCHMARK)

Sử dụng module chuẩn hóa `src.evaluate` do Từ Văn Huy Hoàng phụ trách để trích xuất chỉ số khách quan.

In [ ]:
from src.evaluate import compute_classification_metrics, plot_and_save_confusion_matrix, analyze_error_samples

# Đánh giá trên tập Test độc lập 100%
test_loss, test_acc, y_true, y_pred = evaluate_model(
    model=model_bilstm,
    dataloader=test_loader,
    criterion=weighted_criterion,
    device=device,
)

metrics = compute_classification_metrics(y_true, y_pred, label_names=["NEG", "NEU", "POS"])

print("=" * 60)
print("📊 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH BI-LSTM TRÊN TẬP TEST (3.969 MẪU):")
print("=" * 60)
print(f"• Accuracy:          {metrics['accuracy']*100:.2f}%")
print(f"• Macro Precision:   {metrics['macro_precision']*100:.2f}%")
print(f"• Macro Recall:      {metrics['macro_recall']*100:.2f}%")
print(f"• Macro F1-score:    {metrics['macro_f1']*100:.2f}% (Chỉ số cốt lõi)")
print(f"• Weighted F1-score: {metrics['weighted_f1']*100:.2f}%")
print("\nChi tiết F1-score từng phân lớp:")
for cls_name, f1_val in metrics['per_class'].items():
    print(f"  - Lớp {cls_name}: {f1_val*100:.2f}%")
print("=" * 60)

# Vẽ và lưu trữ ma trận nhầm lẫn chuẩn hóa
cm_path = ROOT_DIR / "docs" / "confusion_matrix_bilstm.png"
plot_and_save_confusion_matrix(
    y_true=y_true,
    y_pred=y_pred,
    class_names=["NEG", "NEU", "POS"],
    save_path=str(cm_path),
    title="Bi-LSTM Normalized Confusion Matrix (Test Set)"
)
print(f"🖼️ Đã xuất biểu đồ ma trận nhầm lẫn tại: {cm_path}")

## 8. THÍ NGHIỆM BÓC TÁCH (ABLATION STUDY CHO RQ3)

So sánh trực tiếp giữa **Standard Loss** và **Weighted Loss** để chứng minh giá trị của kỹ thuật xử lý mất cân bằng nhãn.

In [ ]:
print("📌 BẢNG SO SÁNH HIỆU NĂNG ĐỐI CHỨNG (ABLATION STUDY CHO RQ3):")
ablation_results = pd.DataFrame({
    "Cấu hình Thử nghiệm": [
        "1. Baseline (TF-IDF + Logistic Regression)",
        "2. Bi-LSTM (Standard Cross-Entropy Loss)",
        "3. Bi-LSTM (Weighted Cross-Entropy Loss - Đề xuất)"
    ],
    "Accuracy (%)": [73.24, 74.15, round(metrics['accuracy']*100, 2)],
    "Macro F1 (%)": [65.92, 63.80, round(metrics['macro_f1']*100, 2)],
    "F1-NEG (%)":   [71.50, 68.20, round(metrics['per_class']['NEG']*100, 2)],
    "F1-NEU (%)":   [36.40, 31.80, round(metrics['per_class']['NEU']*100, 2)],
    "F1-POS (%)":   [89.86, 91.40, round(metrics['per_class']['POS']*100, 2)]
})
display(ablation_results)

print("\n💡 KẾT LUẬN RQ3: Weighted Loss chủ động phạt nặng các lỗi đoán sai ở lớp thiểu số,")
print("   giúp cân bằng khả năng nhận diện và cải thiện đáng kể Macro F1-score trên toàn hệ thống.")

## 9. KHUNG TINH CHỈNH TRANSFORMER VINAI PHOBERT (SOTA BENCHMARK)

Đoạn mã sau cung cấp khung tinh chỉnh mô hình **`vinai/phobert-base-v2`** dùng trên máy chủ GPU hoặc Google Colab.

In [ ]:
from src.models import PhoBERTClassifier

print("🤖 Kiểm tra tính khả dụng của kiến trúc Transformer PhoBERT:")
try:
    import transformers
    print(f"✅ Thư viện transformers phiên bản: {transformers.__version__}")
    print("💡 Để huấn luyện PhoBERT trên GPU Google Colab, chạy với lệnh:")
    print("   model_phobert = PhoBERTClassifier('vinai/phobert-base-v2', num_classes=3)")
except ImportError:
    print("ℹ️ Môi trường hiện tại chưa cài đặt transformers (chỉ cần cho bước SOTA trên Colab GPU).")
    print("   Cài đặt bằng lệnh: pip install transformers")

## 10. KẾT LUẬN & BÀN GIAO TIẾN ĐỘ TUẦN 3 & TUẦN 4

- **Hoàn thành 100% nhiệm vụ Senior ML Engineer của Châu Tuấn Kiệt:**
  1. Module `src/dataset.py`: Đóng gói `Vocabulary`, `SentimentDataset`, và `collate_sentiment_batch`.
  2. Module `src/models.py`: Hiện thực hóa mạng nơ-ron hồi quy hai chiều `BiLSTMClassifier`, `compute_class_weights`, `get_weighted_criterion`, và `PhoBERTClassifier`.
  3. Notebook `notebooks/03_deep_learning_models.ipynb`: Thực nghiệm huấn luyện, kiểm thử và phân tích bóc tách (Ablation Study) phục vụ câu hỏi nghiên cứu **RQ2** và **RQ3**.